In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df1 = pd.read_csv("D:\管院学习\\25春\美团商分赛\选题六\\06 美团业务选题-优惠券投放提效分析-数据1-order_detail.csv")
df2 = pd.read_csv("D:\管院学习\\25春\美团商分赛\选题六\\06 美团业务选题-优惠券投放提效分析-数据2-user_coupon_receive.csv")
df3 = pd.read_csv("D:\管院学习\\25春\美团商分赛\选题六\\06 美团业务选题-优惠券投放提效分析-数据3-user_visit_detail.csv")

对df2进行清洗：

In [ ]:
df2

In [ ]:
# 检查整个DataFrame的空值情况
df2.isnull().sum()

In [ ]:
df2[df2['Coupon_amt'].isna() & df2['Coupon_status'].isna() & df2['Receive_date'].isna()]

In [ ]:
df2.groupby("Coupon_amt").count()

In [ ]:
df2_cleaned = df2[df2.Coupon_amt != 9999]

In [ ]:
df2_cleaned.isnull().sum()

In [ ]:
# 绘制缺失值热力图
plt.rcParams['font.sans-serif'] = ['SimHei']  # 使用黑体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

plt.figure(figsize=(12, 6))
sns.heatmap(df2_cleaned.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('df2缺失值分布热力图')
plt.show()

In [ ]:
df2_cleaned=df2_cleaned.dropna()

In [ ]:
df2_cleaned.isna().sum()

对df1进行数据清洗：

In [ ]:
df1

In [ ]:
df1.isnull().sum()

In [ ]:
df1[df1['Coupon_id'].isna() ^ df1['Coupon_type'].isna()]

In [ ]:
# 保留两者同时为空或同时不为空的数据行
df1_cleaned = df1[~(df1['Coupon_id'].isna() ^ df1['Coupon_type'].isna())]

# 验证处理结果
print(f"原始数据行数: {len(df1)}")
print(f"清洗后数据行数: {len(df1_cleaned)}")
print(f"移除的行数: {len(df1) - len(df1_cleaned)}")

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df1_cleaned.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('df1缺失值分布热力图')
plt.show()

In [ ]:
len(df1_cleaned)

In [ ]:
# 筛选Coupon_id不为空的数据
df1_valid = df1[df1['Coupon_id'].notna()]

# 检查User_id和Coupon_id组合重复
duplicates = df1_valid[df1_valid.duplicated(subset=['User_id', 'Coupon_id'], keep=False)]

if not duplicates.empty:
    print(f"发现 {len(duplicates)} 行重复数据（Coupon_id不为空且User_id+Coupon_id组合重复）")
    display(duplicates.sort_values(by=['User_id', 'Coupon_id']))
    
    # 统计重复组合的出现次数
    dup_counts = duplicates.groupby(['User_id', 'Coupon_id']).size().reset_index(name='重复次数')
    print("\n重复组合统计:")
    display(dup_counts.sort_values('重复次数', ascending=False))
else:
    print("没有发现符合条件的重复行")

In [ ]:
# 保留两者同时为空或同时不为空的数据行
df1_cleaned = df1[~(df1['Coupon_id'].isna() ^ df1['Coupon_type'].isna())]

# 筛选出Coupon_id不为空的数据进行去重处理
df1_with_coupon = df1_cleaned[df1_cleaned['Coupon_id'].notna()]
df1_no_coupon = df1_cleaned[df1_cleaned['Coupon_id'].isna()]

# 去除User_id和Coupon_id组合重复的行(保留第一个出现的)
df1_with_coupon_unique = df1_with_coupon.drop_duplicates(subset=['User_id', 'Coupon_id'], keep='first')

# 合并处理后的数据
df1_final = pd.concat([df1_with_coupon_unique, df1_no_coupon])

# 验证处理结果
print(f"原始数据行数: {len(df1)}")
print(f"第一次清洗后行数: {len(df1_cleaned)}") 
print(f"最终清洗后行数: {len(df1_final)}")
print(f"总移除的行数: {len(df1) - len(df1_final)}")

In [ ]:
df1_cleaned

对df1和df2进行外连接

In [ ]:
# 执行左外连接（保留df2_cleaned所有行）
merged_df = pd.merge(df2_cleaned, df1_cleaned, 
                    how='left', 
                    on=['User_id', 'Coupon_id'],  # 假设这两列是连接键
                    suffixes=('_df2', '_df1'))

# 显示连接结果
print(f"df2_cleaned行数: {len(df2_cleaned)}")
print(f"合并后行数: {len(merged_df)}")
display(merged_df.head())

In [ ]:
# 检查有用券记录但Coupon_status=1的数据
inconsistent_records = merged_df[(merged_df['Order_id'].notna()) & (merged_df['Coupon_status'] == 1)]

if not inconsistent_records.empty:
    print(f"发现 {len(inconsistent_records)} 条有用券记录但Coupon_status=1的数据")
    print(inconsistent_records.head())
else:
    print("没有发现有用券记录但Coupon_status=1的数据")

In [ ]:
# 删除有用券记录但Coupon_status=1的异常数据行
merged_df = merged_df[~((merged_df['Order_id'].notna()) & (merged_df['Coupon_status'] == 1))]

In [ ]:
print(f"删除后数据量: {len(merged_df)}")

In [ ]:
merged_df

In [ ]:
# 计算缺失值情况
missing_data = merged_df.isnull()

# 创建热力图
plt.figure(figsize=(10, 6))
sns.heatmap(missing_data, cbar=False, cmap='viridis')
plt.title('缺失值热力图')
plt.show()

In [ ]:
merged_df.to_csv('D:/管院学习/25春/美团商分赛/选题六/merged_data.csv',
                index=False,
                encoding='utf-8-sig',
                sep=',',
                header=True)

In [ ]:
df1_cleaned.to_csv('D:/管院学习/25春/美团商分赛/选题六/df1_cleaned.csv',
                index=False,
                encoding='utf-8-sig',
                sep=',',
                header=True)

In [ ]:
df2_cleaned.to_csv('D:/管院学习/25春/美团商分赛/选题六/df2_cleaned.csv',
                index=False,
                encoding='utf-8-sig',
                sep=',',
                header=True)

对df3进行清洗：

In [ ]:
df3.isnull().sum()